# Arabic E-Commerce Sentiment Analysis

## Objective

Build a three-class Arabic sentiment classifier for e-commerce product reviews.

The model will classify each review as:

- Positive
- Neutral
- Negative

The final production model will use AraBERT and will later be integrated into the MLOps pipeline described in the course handbook.

## Dataset split

- `train.csv` → training
- `dev.csv` → validation
- `test.csv` → final held-out test

The predefined dataset splits will be preserved.

# Imports

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt

# Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "train.csv"
DEV_PATH = DATA_DIR / "dev.csv"
TEST_PATH = DATA_DIR / "test.csv"

TRAIN_PATH, DEV_PATH, TEST_PATH

(WindowsPath('e:/03 Education/Online Courses/MLOps MENA/Final Project/arabic-sentiment-mlops/data/raw/train.csv'),
 WindowsPath('e:/03 Education/Online Courses/MLOps MENA/Final Project/arabic-sentiment-mlops/data/raw/dev.csv'),
 WindowsPath('e:/03 Education/Online Courses/MLOps MENA/Final Project/arabic-sentiment-mlops/data/raw/test.csv'))

# Load the three official splits

In [ ]:
train = pd.read_csv(TRAIN_PATH)
dev = pd.read_csv(DEV_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Dev:", dev.shape)
print("Test:", test.shape)

Train: (87333, 2)
Dev: (9990, 2)
Test: (10014, 2)


In [4]:
train.head()

,حديد خفيف بس تودي الغرض بغرفة الغسيل اعجبتني بنتظيم وترتيب,1
0,ريبات وحجمم مو صغير حيل انا توقعتا صغير بس حلو...,0
1,رائعة و مناسبة للتخزين,2
2,حبييييت مرا حبيييي عجزت عن الوصف شكرا نون,0
3,تجنن اخذتا ل القو تجي حبتين,0
4,جميل جداً والتوصيل سريع وشوفو المقاسات قبل الطلب,1


In [5]:
dev.head()

,تطبيق رائع جدا وطريقة عرض الحروف ممتعة ومبسطة للأطفال وبيعلم الطفل يكتب الحرف باسل طريقة ممكنة,2
0,ممتاز وآمن للأطفال أفضل من كثير من التطبيقات ...,2
1,جدا جميل وانصح اي احد يحمل ذا البرنامج,2
2,تعليمية للأطفال رائعة جدا جدا جدا,2
3,لعبة جميلة ورائعة وتعلم الطفل أسرع على كتابة ا...,2
4,لعب جميل للأطفال جميل جدا جدا جدا,2


In [6]:
test.head()

,لعبة أكثر من رائعة لكنا بطيئة بالتحميل ودائما يكتب انت تلعب بدون نت رغم النت القوي أتمنى حل ذ المشكلة وأيضاً أتمنى ان استطيع وضع اعجاب لأي بلدة اعجبتني ومساعدة بعضنا وألف شكرررر,2
0,ولل روع و أفضل لعب بناء شفتا وتستال وبالمناسبة...,2
1,ممتازة جداً انابستمتع جدا في لعبا ولكن لماذا د...,2
2,ممتازة جدا ورائعة اتمنى ان نستطيع التواصل مع ا...,2
3,السلام عليكم العلبة مميزة و رائعة من فضلكم أ...,2
4,لعبة جداجداجدا راااااائعة انصح بتحميلا ولكن نا...,2


# Schema

In [7]:
print("Train columns:")
print(train.columns.tolist())

print("\nTrain dtypes:")
print(train.dtypes)

Train columns:
['حديد خفيف بس تودي الغرض بغرفة الغسيل اعجبتني بنتظيم وترتيب', '1']

Train dtypes:
حديد خفيف بس تودي الغرض بغرفة الغسيل اعجبتني بنتظيم وترتيب      str
1                                                             int64
dtype: object


In [10]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 87333 entries, 0 to 87332
Data columns (total 2 columns):
 #   Column                                                      Non-Null Count  Dtype
---  ------                                                      --------------  -----
 0   حديد خفيف بس تودي الغرض بغرفة الغسيل اعجبتني بنتظيم وترتيب  87333 non-null  str  
 1   1                                                           87333 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.3 MB


In [11]:
"""
I wrote names=["text", "label"] Because the DataFrame has no header at all so it already lost the first review from the data by treating it as the header.

So we should reload the CSV correctly rather than rename the damaged DataFrame.
"""

train = pd.read_csv(
    TRAIN_PATH,
    header=None,
    names=["text", "label"]
)

dev = pd.read_csv(
    DEV_PATH,
    header=None,
    names=["text", "label"]
)

test = pd.read_csv(
    TEST_PATH,
    header=None,
    names=["text", "label"]
)

In [12]:
print("Train:", train.shape)
print("Dev:", dev.shape)
print("Test:", test.shape)

Train: (87334, 2)
Dev: (9991, 2)
Test: (10015, 2)


In [13]:
train.head()

,text,label
0,حديد خفيف بس تودي الغرض بغرفة الغسيل اعجبتني ب...,1
1,ريبات وحجمم مو صغير حيل انا توقعتا صغير بس حلو...,0
2,رائعة و مناسبة للتخزين,2
3,حبييييت مرا حبيييي عجزت عن الوصف شكرا نون,0
4,تجنن اخذتا ل القو تجي حبتين,0


In [14]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 87334 entries, 0 to 87333
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    87334 non-null  str  
 1   label   87334 non-null  int64
dtypes: int64(1), str(1)
memory usage: 1.3 MB


No Missing values

# Labels

In [15]:
train["label"].value_counts()

label
0    36782
1    25439
2    25113
Name: count, dtype: int64

In [ ]:
print({train["label"].value_counts(normalize=True)})
print("==========================================")
print(dev["label"].value_counts(normalize=True))
print("==========================================")
print(test["label"].value_counts(normalize=True))

label
0    0.421165
1    0.291284
2    0.287551
Name: proportion, dtype: float64
label
0    0.334101
1    0.333500
2    0.332399
Name: proportion, dtype: float64
label
2    0.333899
0    0.333400
1    0.332701
Name: proportion, dtype: float64


# Arabic text inspection

In [21]:
pd.set_option("display.max_colwidth", 300)

train.sample(10, random_state=42)

,text,label
28510,جيد تعامل الاستقبال ممتازالنت ممتاز عدم الرد على الاتف ولابد من النزول للاستقبال خدمات الغرف جدا سيئ,1
24254,عمل ممتاز وراقي كالعادة اتمنالك الخير,2
81825,نار,0
8518,كتاب يعطي امل وتفائل انصح ب,2
50856,ذا الرواية تعتبر بالنسبة لي تأريخا لموطني السليب حيك بمارة أدبية رصينة لدرجة مبكية لقلوبنا قلوبنا الكسيرة أبدعت الكاتبة المصرية رضوى عاشور في ذا العمل الروائي التأريخي و الذي يعتبر أيقونة في التاريخ الفلسطيني عايشناا عن قرب فأسرتنا رغما عنا أزمة و وصمة عار في وجو شتى العربي منا قبل الدو...,0
86963,تموت وكبير مرا,0
11409,الل يوفقك استاذ عبدالرحمن اني,2
5026,رااااائع الزيارة القادمة بنفس الفندق باذن الل اسعار الماكولات والمشروبات مبالغ فيا,2
24619,جميل جدا ورائحت تغطي المكان,2
72261,عبارات احلام قاتلة مميزةفريدة من نوعا مؤلمة ايضا و عنوان الكتاب يليق تماما بما يدور في بالا من افكار,0


In [22]:
for text in train["text"].sample(10, random_state=42):
    print(text)
    print("-" * 80)

جيد تعامل الاستقبال ممتازالنت ممتاز عدم الرد على الاتف ولابد من النزول للاستقبال خدمات الغرف جدا سيئ
--------------------------------------------------------------------------------
عمل ممتاز وراقي  كالعادة اتمنالك الخير
--------------------------------------------------------------------------------
نار
--------------------------------------------------------------------------------
كتاب يعطي امل وتفائل انصح ب
--------------------------------------------------------------------------------
ذا الرواية  تعتبر بالنسبة لي تأريخا لموطني السليب  حيك بمارة أدبية رصينة  لدرجة مبكية لقلوبنا  قلوبنا الكسيرة  أبدعت الكاتبة المصرية رضوى عاشور في ذا العمل الروائي التأريخي و الذي يعتبر أيقونة في التاريخ الفلسطيني  عايشناا عن قرب  فأسرتنا رغما عنا  أزمة و وصمة عار في وجو شتى  العربي منا قبل الدولي  كم أتمنى أن أتنفس واك يا فلسطين  يا موطني  الجلال و الجمال  ذا الرواية من حقا أن ترص انب أعمال إبرايم نصر الل التي اقتنيتا  استمتعت و كدت أن أتور في أثناء قراءتي لذ الرواية  شكرا لرضوى عاشور على ذا الأسلو

In [ ]:
# Find the actual label meaning

for label in sorted(train["label"].unique()):
    print(f"\n===== LABEL {label} =====")
    samples = train.loc[train["label"] == label, "text"].sample(
        10, random_state=42
    )
    for text in samples:
        print("-", text)


===== LABEL 0 =====
- مخيب للأمل الموقع فقط نظافة الغرفة
- ضعيف جدا  ليست غرفتي التي حجزتا
- ضعيف لا شي ذا فندق وشقق مفروش الاثاث قديم والاصنصير خربتن والفندق لا يوجد ب نظافة وريحتة خايس
- ما أزال
- مقبول قرب من مول الإمارات انقطاع الماء  غير ان موظفين الخدمات غير متعاونين
- سيء   الغرف سيئة ولايوجد عازل للصوت عدد موظفي الخدمة قليل يتطلب حضورم وقت طويل لا يرقى لمستوى اكثر من نجمتين
- الخصوصي مفقود لأسرة محافظ   النظاف والخصوصي فقط في المسبح اما الشاطيء فو مفتوح
- شكرا لك يا محمد والل يعطيك العافية
- كتاب خفيف أم ما خرجت ب من أن الإنسان بحاجة للعزلو والتأمل ليشحذ طاقات روح
- محمد يتم بالتفاصيل وسريع بالرد دائما يحاول ان يجد طريقة لفم الطرف الآخر وقد سلمني طلبي خلال اقل من يومين

===== LABEL 1 =====
- جيد  الفندق مناسب جدا للعوائل القيم مقابل المال ساعات سباح للنساءانترنت مجانيموقع الفندق مقابل صحارى مول المواقف ضيق جدا جدا صغر مساحة بعض الغرف عزل الصوت قليل
- فندق جميل والراح في عنوان ذا الفندق  دوء المكان ومافي الي ازعاج العلام الخاص بالتجا القبل كان موقعا غير صحيح
- مريح وقريب من الح

In [ ]:
# Check duplicates

print("Duplicate texts in train:", train["text"].duplicated().sum())
print("Duplicate texts in dev:", dev["text"].duplicated().sum())
print("Duplicate texts in test:", test["text"].duplicated().sum())

Duplicate texts in train: 11998
Duplicate texts in dev: 44
Duplicate texts in test: 0


In [ ]:
# Check cross-split leakage

train_texts = set(train["text"])
dev_texts = set(dev["text"])
test_texts = set(test["text"])

print("Train ∩ Dev:", len(train_texts & dev_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Dev ∩ Test:", len(dev_texts & test_texts))

Train ∩ Dev: 7922
Train ∩ Test: 1195
Dev ∩ Test: 2


In [26]:
# Check missing and empty text

for name, df in {
    "train": train,
    "dev": dev,
    "test": test,
}.items():
    empty = df["text"].fillna("").str.strip().eq("").sum()
    missing = df["text"].isna().sum()

    print(name)
    print("  missing:", missing)
    print("  empty:", empty)

train
  missing: 0
  empty: 0
dev
  missing: 0
  empty: 0
test
  missing: 0
  empty: 0


In [27]:
# Check the total and split proportions

total = len(train) + len(dev) + len(test)

print("Total:", total)

for name, df in {
    "train": train,
    "dev": dev,
    "test": test,
}.items():
    print(f"{name}: {len(df):,} ({len(df) / total:.2%})")

Total: 107340
train: 87,334 (81.36%)
dev: 9,991 (9.31%)
test: 10,015 (9.33%)
